<a href="https://colab.research.google.com/github/anamelbsi/projeto-ic-segmentacao/blob/main/segmentacao_corpos_dagua/SAM2/SAM2%20corpos%20d'%C3%A1gua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# bloco 1
!pip install opencv-python torch torchvision matplotlib tqdm
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

import os
import cv2
import numpy as np
import torch
import zipfile
from tqdm import tqdm
from google.colab import drive
import matplotlib.pyplot as plt

drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'usando dispositivo: {device}')

!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

sam2 = build_sam2(model_cfg, checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2)

print('modelo sam2 carregado')

caminho_zip = "/content/drive/MyDrive/waterbodiesdataset.zip"
with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')
print('dataset carregado')

pasta_imagens = '/content/dataset/Water Bodies Dataset/Images'
pasta_mascaras = '/content/dataset/Water Bodies Dataset/Masks'

imagens = sorted([f for f in os.listdir(pasta_imagens) if f.endswith(('.png', '.jpg', '.jpeg'))])
mascaras = sorted([f for f in os.listdir(pasta_mascaras) if f.endswith(('.png', '.jpg', '.jpeg'))])

print(f'imagens: {len(imagens)}')
print(f'mascaras: {len(mascaras)}')

def calcular_metricas(mascara_pred, mascara_gt):
    pred = mascara_pred > 0.5
    gt = mascara_gt > 0.5

    intersecao = np.logical_and(pred, gt).sum()
    uniao = np.logical_or(pred, gt).sum()

    total_pixels = pred.size
    acertos = (pred == gt).sum()
    acuracia = acertos / total_pixels

    iou = intersecao / uniao if uniao > 0 else 0

    return acuracia, iou

acuracia_total = []
iou_total = []

for img_nome, mask_nome in tqdm(zip(imagens, mascaras), total=len(imagens)):
    img_path = os.path.join(pasta_imagens, img_nome)
    mask_path = os.path.join(pasta_mascaras, mask_nome)

    imagem = cv2.imread(img_path)
    imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

    mascara_gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mascara_gt = mascara_gt / 255.0

    predictor.set_image(imagem)

    h, w = imagem.shape[:2]
    ponto = np.array([[w//2, h//2]])
    rotulo = np.array([1])

    mascaras_pred, scores, logits = predictor.predict(
        point_coords=ponto,
        point_labels=rotulo,
        multimask_output=False
    )

    mascara_pred = mascaras_pred[0]

    if mascara_pred.shape != mascara_gt.shape:
        mascara_pred = cv2.resize(mascara_pred, (mascara_gt.shape[1], mascara_gt.shape[0]))

    acc, iou = calcular_metricas(mascara_pred, mascara_gt)
    acuracia_total.append(acc)
    iou_total.append(iou)

    print(f'{img_nome}: acc={acc:.4f}, iou={iou:.4f}')

print(f'\nresultados finais sam2:')
print(f'acuracia media: {np.mean(acuracia_total):.4f}')
print(f'iou medio: {np.mean(iou_total):.4f}')
print(f'desvio padrao acuracia: {np.std(acuracia_total):.4f}')
print(f'desvio padrao iou: {np.std(iou_total):.4f}')

  Cloning https://github.com/facebookresearch/segment-anything-2.git to /tmp/pip-req-build-2yd3n_50
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything-2.git /tmp/pip-req-build-2yd3n_50
  Resolved https://github.com/facebookresearch/segment-anything-2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4


In [ ]:
# bloco 2
# especificação
import pandas as pd
from datetime import datetime


print('especificação dos resultados - sam2 no dataset water bodies')


print('\n1. informações gerais:')
print(f'   total de imagens processadas: {len(imagens)}')
print(f'   dispositivo utilizado: {device}')
print(f'   modelo sam2: hiera_large')
print(f'   tipo de prompt: ponto central')

print('\n2. métricas principais:')
print(f'   acurácia média: {np.mean(acuracia_total):.4f} ({np.mean(acuracia_total)*100:.2f}%)')
print(f'   iou médio: {np.mean(iou_total):.4f} ({np.mean(iou_total)*100:.2f}%)')
print(f'   desvio padrão acurácia: {np.std(acuracia_total):.4f}')
print(f'   desvio padrão iou: {np.std(iou_total):.4f}')

print('\n3. percentis:')
percentis = [25, 50, 75, 90, 95]
print(f'   percentis iou:')
for p in percentis:
    valor = np.percentile(iou_total, p)
    print(f'      {p}%: {valor:.4f}')

print('\n4. distribuição por faixas de iou:')
faixas = [(0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
for min_f, max_f in faixas:
    count = sum(1 for iou in iou_total if min_f <= iou < max_f)
    pct = count / len(iou_total) * 100
    print(f'   {min_f:.1f} - {max_f:.1f}: {count} imagens ({pct:.1f}%)')

print('\n5. top 5 melhores iou:')
melhores_indices = np.argsort(iou_total)[-5:][::-1]
for i, idx in enumerate(melhores_indices, 1):
    print(f'   {i}. {imagens[idx]}: iou={iou_total[idx]:.4f}')

print('\n6. top 5 piores iou:')
piores_indices = np.argsort(iou_total)[:5]
for i, idx in enumerate(piores_indices, 1):
    print(f'   {i}. {imagens[idx]}: iou={iou_total[idx]:.4f}')

# salvar resultados
timestamp = datetime.now().strftime('%y%m%d_%H%M%S')
arquivo_csv = f'resultados_sam2_{timestamp}.csv'

with open(arquivo_csv, 'w', newline='') as f:
    import csv
    escritor = csv.writer(f)
    escritor.writerow(['imagem', 'acuracia', 'iou'])
    for i, img_nome in enumerate(imagens):
        escritor.writerow([img_nome, f'{acuracia_total[i]:.4f}', f'{iou_total[i]:.4f}'])

print(f'\nresultados salvos em: {arquivo_csv}')

from google.colab import files
files.download(arquivo_csv)

In [ ]:
# bloco 3 - comparação sam1 vs sam2
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from google.colab import files

# resultados do sam1 (completos)
# estatísticas descritivas
acuracia_media_sam1 = 0.6806
iou_media_sam1 = 0.4273
std_acuracia_sam1 = 0.2944
std_iou_sam1 = 0.3698

# percentis iou sam1
percentis_iou_sam1 = {
    25: 0.0115,
    50: 0.4235,
    75: 0.7998,
    90: 0.9334,
    95: 0.9637
}

# percentis acurácia sam1
percentis_acc_sam1 = {
    25: 0.4718,
    50: 0.7875,
    75: 0.9326,
    90: 0.9758,
    95: 0.9868
}

# distribuição iou sam1
faixas = [(0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
dist_iou_sam1 = [1131, 253, 319, 428, 709]
dist_acc_sam1 = [287, 279, 405, 494, 1375]

# resumo consolidado sam1
resumo_sam1 = {
    'iou_05': 1298,
    'iou_07': 933,
    'iou_08': 710,
    'iou_03': 1262
}

print('comparação sam1 vs sam2')

print('\n1. métricas principais:')
print(f'métrica              | sam1      | sam2      | diferença')
print(f'acurácia média       | {acuracia_media_sam1:.4f}   | {np.mean(acuracia_total):.4f}   | {np.mean(acuracia_total) - acuracia_media_sam1:+.4f}')
print(f'iou médio            | {iou_media_sam1:.4f}   | {np.mean(iou_total):.4f}   | {np.mean(iou_total) - iou_media_sam1:+.4f}')
print(f'desvio acurácia      | {std_acuracia_sam1:.4f}   | {np.std(acuracia_total):.4f}   | {np.std(acuracia_total) - std_acuracia_sam1:+.4f}')
print(f'desvio iou           | {std_iou_sam1:.4f}   | {np.std(iou_total):.4f}   | {np.std(iou_total) - std_iou_sam1:+.4f}')

print('\n2. percentis iou:')
print(f'percentil | sam1      | sam2      | diferença')
for p in [25, 50, 75, 90, 95]:
    valor_sam2 = np.percentile(iou_total, p)
    print(f'{p}%        | {percentis_iou_sam1[p]:.4f}   | {valor_sam2:.4f}   | {valor_sam2 - percentis_iou_sam1[p]:+.4f}')

print('\n3. percentis acurácia:')
print('-'*50)
for p in [25, 50, 75, 90, 95]:
    valor_sam2 = np.percentile(acuracia_total, p)
    print(f'{p}%        | {percentis_acc_sam1[p]:.4f}   | {valor_sam2:.4f}   | {valor_sam2 - percentis_acc_sam1[p]:+.4f}')

print('\n4. distribuição iou por faixa:')
print(f'faixa      | sam1               | sam2               | diferença')
for i, (min_f, max_f) in enumerate(faixas):
    count_sam2 = sum(1 for iou in iou_total if min_f <= iou < max_f)
    pct_sam1 = dist_iou_sam1[i] / 2841 * 100
    pct_sam2 = count_sam2 / len(iou_total) * 100
    print(f'{min_f:.1f}-{max_f:.1f}   | {dist_iou_sam1[i]:4d} ({pct_sam1:5.1f}%) | {count_sam2:4d} ({pct_sam2:5.1f}%) | {count_sam2 - dist_iou_sam1[i]:+4d}')

print('\n5. resumo consolidado:')
print(f'critério        | sam1               | sam2               | diferença')

iou_05_sam2 = len([iou for iou in iou_total if iou >= 0.5])
iou_07_sam2 = len([iou for iou in iou_total if iou >= 0.7])
iou_08_sam2 = len([iou for iou in iou_total if iou >= 0.8])
iou_03_sam2 = len([iou for iou in iou_total if iou < 0.3])

print(f'iou >= 0.5      | {resumo_sam1["iou_05"]:4d} ({resumo_sam1["iou_05"]/2841*100:5.1f}%) | {iou_05_sam2:4d} ({iou_05_sam2/len(iou_total)*100:5.1f}%) | {iou_05_sam2 - resumo_sam1["iou_05"]:+4d}')
print(f'iou >= 0.7      | {resumo_sam1["iou_07"]:4d} ({resumo_sam1["iou_07"]/2841*100:5.1f}%) | {iou_07_sam2:4d} ({iou_07_sam2/len(iou_total)*100:5.1f}%) | {iou_07_sam2 - resumo_sam1["iou_07"]:+4d}')
print(f'iou >= 0.8      | {resumo_sam1["iou_08"]:4d} ({resumo_sam1["iou_08"]/2841*100:5.1f}%) | {iou_08_sam2:4d} ({iou_08_sam2/len(iou_total)*100:5.1f}%) | {iou_08_sam2 - resumo_sam1["iou_08"]:+4d}')
print(f'iou < 0.3       | {resumo_sam1["iou_03"]:4d} ({resumo_sam1["iou_03"]/2841*100:5.1f}%) | {iou_03_sam2:4d} ({iou_03_sam2/len(iou_total)*100:5.1f}%) | {iou_03_sam2 - resumo_sam1["iou_03"]:+4d}')

print('\n6. análise de melhoria:')
melhoria_iou = np.mean(iou_total) - iou_media_sam1
melhoria_acc = np.mean(acuracia_total) - acuracia_media_sam1
print(f'melhoria no iou médio: {melhoria_iou:+.4f} ({melhoria_iou/abs(iou_media_sam1)*100:+.1f}%)')
print(f'melhoria na acurácia média: {melhoria_acc:+.4f} ({melhoria_acc/abs(acuracia_media_sam1)*100:+.1f}%)')

# criar gráficos comparativos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# gráfico 1: comparação iou médio
modelos = ['sam1', 'sam2']
iou_medias = [iou_media_sam1, np.mean(iou_total)]
bars = axes[0, 0].bar(modelos, iou_medias, color=['orange', 'blue'])
axes[0, 0].set_ylabel('iou médio')
axes[0, 0].set_title('comparação iou médio')
axes[0, 0].set_ylim(0, 1)
for bar, val in zip(bars, iou_medias):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.3f}', ha='center')

# gráfico 2: comparação distribuição iou
x = np.arange(len(faixas))
width = 0.35
pct_sam1 = [count/2841*100 for count in dist_iou_sam1]
pct_sam2 = [sum(1 for iou in iou_total if min_f <= iou < max_f)/len(iou_total)*100 for min_f, max_f in faixas]
axes[0, 1].bar(x - width/2, pct_sam1, width, label='sam1', color='orange')
axes[0, 1].bar(x + width/2, pct_sam2, width, label='sam2', color='blue')
axes[0, 1].set_xlabel('faixas de iou')
axes[0, 1].set_ylabel('percentual de imagens (%)')
axes[0, 1].set_title('distribuição iou')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([f'{min_f:.1f}-{max_f:.1f}' for min_f, max_f in faixas])
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# gráfico 3: comparação percentis iou
percentis_nomes = ['25%', '50%', '75%', '90%', '95%']
valores_sam1 = [percentis_iou_sam1[p] for p in [25, 50, 75, 90, 95]]
valores_sam2 = [np.percentile(iou_total, p) for p in [25, 50, 75, 90, 95]]

x = np.arange(len(percentis_nomes))
axes[1, 0].plot(x, valores_sam1, 'o-', label='sam1', color='orange', linewidth=2, markersize=8)
axes[1, 0].plot(x, valores_sam2, 's-', label='sam2', color='blue', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('percentis')
axes[1, 0].set_ylabel('iou')
axes[1, 0].set_title('comparação percentis iou')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(percentis_nomes)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# gráfico 4: melhoria por percentil
melhorias = [valores_sam2[i] - valores_sam1[i] for i in range(len(valores_sam1))]
colors = ['green' if m > 0 else 'red' for m in melhorias]
axes[1, 1].bar(percentis_nomes, melhorias, color=colors)
axes[1, 1].set_xlabel('percentis')
axes[1, 1].set_ylabel('melhoria no iou (sam2 - sam1)')
axes[1, 1].set_title('melhoria por percentil')
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# salvar comparação
timestamp = datetime.now().strftime('%y%m%d_%H%M%S')
arquivo_comp = f'comparacao_sam1_sam2_{timestamp}.txt'

with open(arquivo_comp, 'w') as f:
    f.write('comparação sam1 vs sam2\n')

    f.write('métricas principais:\n')
    f.write(f'acurácia média: sam1={acuracia_media_sam1:.4f}, sam2={np.mean(acuracia_total):.4f}, dif={np.mean(acuracia_total)-acuracia_media_sam1:+.4f}\n')
    f.write(f'iou médio: sam1={iou_media_sam1:.4f}, sam2={np.mean(iou_total):.4f}, dif={np.mean(iou_total)-iou_media_sam1:+.4f}\n\n')

    f.write('percentis iou:\n')
    for p in [25, 50, 75, 90, 95]:
        f.write(f'{p}%: sam1={percentis_iou_sam1[p]:.4f}, sam2={np.percentile(iou_total, p):.4f}, dif={np.percentile(iou_total, p)-percentis_iou_sam1[p]:+.4f}\n')

    f.write('\nresumo consolidado:\n')
    f.write(f'iou >= 0.5: sam1={resumo_sam1["iou_05"]} ({resumo_sam1["iou_05"]/2841*100:.1f}%), sam2={iou_05_sam2} ({iou_05_sam2/len(iou_total)*100:.1f}%)\n')
    f.write(f'iou >= 0.7: sam1={resumo_sam1["iou_07"]} ({resumo_sam1["iou_07"]/2841*100:.1f}%), sam2={iou_07_sam2} ({iou_07_sam2/len(iou_total)*100:.1f}%)\n')
    f.write(f'iou >= 0.8: sam1={resumo_sam1["iou_08"]} ({resumo_sam1["iou_08"]/2841*100:.1f}%), sam2={iou_08_sam2} ({iou_08_sam2/len(iou_total)*100:.1f}%)\n')
    f.write(f'iou < 0.3: sam1={resumo_sam1["iou_03"]} ({resumo_sam1["iou_03"]/2841*100:.1f}%), sam2={iou_03_sam2} ({iou_03_sam2/len(iou_total)*100:.1f}%)\n')

print(f'\narquivo de comparação salvo: {arquivo_comp}')
files.download(arquivo_comp)

In [ ]:
# bloco 4
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from google.colab import files


try:
    if 'iou_total' not in locals():
        import csv
        arquivo_csv = None
        for f in os.listdir():
            if f.startswith('resultados_sam2_') and f.endswith('.csv'):
                arquivo_csv = f
                break

        if arquivo_csv:
            dados = []
            with open(arquivo_csv, 'r') as file:
                reader = csv.reader(file)
                next(reader)
                for row in reader:
                    dados.append(row)

            imagens = [row[0] for row in dados]
            acuracia_total = [float(row[1]) for row in dados]
            iou_total = [float(row[2]) for row in dados]
            print(f'dados carregados de {arquivo_csv}')
        else:
            print('nenhum arquivo de resultados encontrado')
except:
    print('variáveis não definidas')

# calcular estatísticas do sam2
if 'iou_total' in locals():
    print('especificação dos resultados - sam2 no dataset water bodies')

    print('\n1. informações gerais:')
    print(f'   total de imagens processadas: {len(imagens)}')
    print(f'   dispositivo utilizado: cuda')
    print(f'   modelo sam2: hiera_large')
    print(f'   tipo de prompt: ponto central')

    print('\n2. métricas principais:')
    print(f'   acurácia média: {np.mean(acuracia_total):.4f} ({np.mean(acuracia_total)*100:.2f}%)')
    print(f'   iou médio: {np.mean(iou_total):.4f} ({np.mean(iou_total)*100:.2f}%)')
    print(f'   desvio padrão acurácia: {np.std(acuracia_total):.4f}')
    print(f'   desvio padrão iou: {np.std(iou_total):.4f}')

    print('\n3. percentis:')
    print('   percentis iou:')
    percentis = [25, 50, 75, 90, 95]
    for p in percentis:
        valor = np.percentile(iou_total, p)
        print(f'      {p}%: {valor:.4f}')

    print('\n   percentis acurácia:')
    for p in percentis:
        valor = np.percentile(acuracia_total, p)
        print(f'      {p}%: {valor:.4f}')

    print('\n4. distribuição por faixas de iou:')
    faixas = [(0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
    dist_iou = []
    for min_f, max_f in faixas:
        count = sum(1 for iou in iou_total if min_f <= iou < max_f)
        pct = count / len(iou_total) * 100
        dist_iou.append(count)
        print(f'   {min_f:.1f} - {max_f:.1f}: {count} imagens ({pct:.1f}%)')

    print('\n5. distribuição por faixas de acurácia:')
    dist_acc = []
    for min_f, max_f in faixas:
        count = sum(1 for acc in acuracia_total if min_f <= acc < max_f)
        pct = count / len(acuracia_total) * 100
        dist_acc.append(count)
        print(f'   {min_f:.1f} - {max_f:.1f}: {count} imagens ({pct:.1f}%)')

    print('\n6. top 5 melhores iou:')
    melhores_indices = np.argsort(iou_total)[-5:][::-1]
    for i, idx in enumerate(melhores_indices, 1):
        print(f'   {i}. {imagens[idx]}: iou={iou_total[idx]:.4f}, acc={acuracia_total[idx]:.4f}')

    print('\n7. top 5 piores iou:')
    piores_indices = np.argsort(iou_total)[:5]
    for i, idx in enumerate(piores_indices, 1):
        print(f'   {i}. {imagens[idx]}: iou={iou_total[idx]:.4f}, acc={acuracia_total[idx]:.4f}')

    print('\n8. análise estatística completa:')
    from scipy import stats
    print(f'   count    {len(iou_total)}.000000')
    print(f'   mean     {np.mean(iou_total):.6f}')
    print(f'   std      {np.std(iou_total):.6f}')
    print(f'   min      {np.min(iou_total):.6f}')
    print(f'   25%      {np.percentile(iou_total, 25):.6f}')
    print(f'   50%      {np.percentile(iou_total, 50):.6f}')
    print(f'   75%      {np.percentile(iou_total, 75):.6f}')
    print(f'   max      {np.max(iou_total):.6f}')

    print('\n9. resumo consolidado:')
    iou_05 = len([iou for iou in iou_total if iou >= 0.5])
    iou_07 = len([iou for iou in iou_total if iou >= 0.7])
    iou_08 = len([iou for iou in iou_total if iou >= 0.8])
    iou_03 = len([iou for iou in iou_total if iou < 0.3])

    print(f'   {iou_05} imagens com iou >= 0.5 ({iou_05/len(iou_total)*100:.1f}%)')
    print(f'   {iou_07} imagens com iou >= 0.7 ({iou_07/len(iou_total)*100:.1f}%)')
    print(f'   {iou_08} imagens com iou >= 0.8 ({iou_08/len(iou_total)*100:.1f}%)')
    print(f'   {iou_03} imagens com iou < 0.3 ({iou_03/len(iou_total)*100:.1f}%)')

    # criar gráficos
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(iou_total, bins=20, edgecolor='black', alpha=0.7, color='blue')
    axes[0].axvline(np.mean(iou_total), color='red', linestyle='--', label=f'média: {np.mean(iou_total):.3f}')
    axes[0].set_title('distribuição do iou - sam2')
    axes[0].set_xlabel('iou')
    axes[0].set_ylabel('frequência')
    axes[0].legend()

    axes[1].hist(acuracia_total, bins=20, edgecolor='black', alpha=0.7, color='green')
    axes[1].axvline(np.mean(acuracia_total), color='red', linestyle='--', label=f'média: {np.mean(acuracia_total):.3f}')
    axes[1].set_title('distribuição da acurácia - sam2')
    axes[1].set_xlabel('acurácia')
    axes[1].set_ylabel('frequência')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # salvar especificação
    timestamp = datetime.now().strftime('%y%m%d_%H%M%S')
    arquivo_espec = f'especificacao_sam2_{timestamp}.txt'

    with open(arquivo_espec, 'w') as f:
        f.write('especificação dos resultados - sam2 no dataset water bodies\n')
        f.write('='*60 + '\n\n')

        f.write('1. informações gerais:\n')
        f.write(f'   total de imagens processadas: {len(imagens)}\n')
        f.write(f'   dispositivo utilizado: cuda\n')
        f.write(f'   modelo sam2: hiera_large\n')
        f.write(f'   tipo de prompt: ponto central\n\n')

        f.write('2. métricas principais:\n')
        f.write(f'   acurácia média: {np.mean(acuracia_total):.4f} ({np.mean(acuracia_total)*100:.2f}%)\n')
        f.write(f'   iou médio: {np.mean(iou_total):.4f} ({np.mean(iou_total)*100:.2f}%)\n')
        f.write(f'   desvio padrão acurácia: {np.std(acuracia_total):.4f}\n')
        f.write(f'   desvio padrão iou: {np.std(iou_total):.4f}\n\n')

        f.write('3. percentis:\n')
        f.write('   percentis iou:\n')
        for p in percentis:
            f.write(f'      {p}%: {np.percentile(iou_total, p):.4f}\n')
        f.write('\n   percentis acurácia:\n')
        for p in percentis:
            f.write(f'      {p}%: {np.percentile(acuracia_total, p):.4f}\n\n')

        f.write('4. distribuição por faixas de iou:\n')
        for i, (min_f, max_f) in enumerate(faixas):
            f.write(f'   {min_f:.1f} - {max_f:.1f}: {dist_iou[i]} imagens ({dist_iou[i]/len(iou_total)*100:.1f}%)\n')

        f.write('\n5. distribuição por faixas de acurácia:\n')
        for i, (min_f, max_f) in enumerate(faixas):
            f.write(f'   {min_f:.1f} - {max_f:.1f}: {dist_acc[i]} imagens ({dist_acc[i]/len(acuracia_total)*100:.1f}%)\n')

        f.write('\n6. top 5 melhores iou:\n')
        for i, idx in enumerate(melhores_indices, 1):
            f.write(f'   {i}. {imagens[idx]}: iou={iou_total[idx]:.4f}, acc={acuracia_total[idx]:.4f}\n')

        f.write('\n7. top 5 piores iou:\n')
        for i, idx in enumerate(piores_indices, 1):
            f.write(f'   {i}. {imagens[idx]}: iou={iou_total[idx]:.4f}, acc={acuracia_total[idx]:.4f}\n')

        f.write('\n8. análise estatística completa:\n')
        f.write(f'   count    {len(iou_total)}.000000\n')
        f.write(f'   mean     {np.mean(iou_total):.6f}\n')
        f.write(f'   std      {np.std(iou_total):.6f}\n')
        f.write(f'   min      {np.min(iou_total):.6f}\n')
        f.write(f'   25%      {np.percentile(iou_total, 25):.6f}\n')
        f.write(f'   50%      {np.percentile(iou_total, 50):.6f}\n')
        f.write(f'   75%      {np.percentile(iou_total, 75):.6f}\n')
        f.write(f'   max      {np.max(iou_total):.6f}\n\n')

        f.write('9. resumo consolidado:\n')
        f.write(f'   {iou_05} imagens com iou >= 0.5 ({iou_05/len(iou_total)*100:.1f}%)\n')
        f.write(f'   {iou_07} imagens com iou >= 0.7 ({iou_07/len(iou_total)*100:.1f}%)\n')
        f.write(f'   {iou_08} imagens com iou >= 0.8 ({iou_08/len(iou_total)*100:.1f}%)\n')
        f.write(f'   {iou_03} imagens com iou < 0.3 ({iou_03/len(iou_total)*100:.1f}%)\n')

    print(f'\n\narquivo de especificação salvo: {arquivo_espec}')
    files.download(arquivo_espec)
else:
    print('execute o bloco 1 do sam2 primeiro para gerar os resultados')